In [ ]:
# SkinNova — arms A (stock) + C (image model only) on llm_test (generated by ml/eval/notebooks/make_arms.py)


In [ ]:
%%capture
!pip install -q uv
!uv venv /tmp/ve --python 3.12
!uv pip install --python /tmp/ve/bin/python "litert-lm-api==0.17.1" "numpy==2.5.2" "pillow==12.3.0" scikit-learn pydantic python-dotenv "huggingface-hub==1.33.0"


In [ ]:
import glob, os, subprocess, sys
data = os.path.dirname(glob.glob("/kaggle/input/**/llm_test.jsonl", recursive=True)[0])
need = [data + "/code/ml/eval/eval_arms.py", data + "/llm_eval/llm_test.jsonl"]
print("dataset:", data, {n: os.path.exists(n) for n in need}, flush=True)
assert all(map(os.path.exists, need)), "stale dataset version mounted (no eval_arms / llm_eval) — re-push after the dataset is ready"
os.makedirs("/tmp/d", exist_ok=True)
if not os.path.exists("/tmp/d/llm"): os.symlink(data, "/tmp/d/llm")          # ml.common.paths: LLM_DATA = DATA/llm
subprocess.run(["rm", "-rf", "/kaggle/working/code"]); subprocess.run(["cp", "-r", data + "/code", "/kaggle/working/code"])
from huggingface_hub import hf_hub_download
stock = hf_hub_download("litert-community/gemma-4-E2B-it-litert-lm", "gemma-4-E2B-it.litertlm", revision="b3ca0d2f076785a8f4b2219ddbd2bdb99954eae1", local_dir="/tmp/ref")
env = dict(os.environ, SKINNOVA_DATA_DIR="/tmp/d", PYTHONPATH="/kaggle/working/code")
p = subprocess.Popen(["/tmp/ve/bin/python", "-m", "ml.eval.eval_arms", "--stock", stock, "--arms", "A", "--n", "150", "--threads", "4",
                      "--test-dir", data + "/llm_eval"],
                     cwd="/kaggle/working/code", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if not line.startswith(("INFO", "WARNING", "W0000", "I0000", "E0000")): print(line, end="")
rc = p.wait(); print("exit code", rc)
subprocess.run(["cp", "-r", "/kaggle/working/code/reports", "/kaggle/working/"])
assert rc == 0
